In [ ]:
%%configure -f
{
    "conf": {
        "spark.dynamicAllocation.enabled": "false",
        "spark.driver.cores": "4",
        "spark.driver.memory": "28g",
        "spark.executor.cores": "4",
        "spark.executor.memory": "28g",
        "spark.executor.instances": 1
    }
}

# Profiling â€” Supermarket Net Sales Forecast

**Scenario:** supermarket_net_sales_forecast  
**Generated:** 2026-06-03  

**Key Parameters:**
- Target Variable: TOTAL_NET_SALES
- Date Column: WEEK_START_DT
- Series ID: STORE_LOCATION_ID (50 stores)
- Time Granularity: Weekly (Thursday start)
- min_time_cons: 26 (weekly equivalent of 6 months)

This notebook classifies each store's time series as regular, intermittent, erratic, lumpy, etc. using the enhanced Syntetosâ€“Boylanâ€“Croston (SBC) framework with CVÂ², ADI, and SDDI indicators.

This notebook was customized by the Time Series Forecaster agent.

## 1. Import Required Libraries

In [ ]:
# Core libraries for Fabric
import pandas as pd
import numpy as np
import datetime as dt
import matplotlib.pyplot as plt
from scipy.stats.mstats import winsorize
from pathlib import Path

print("âœ… Core libraries imported successfully")

## 2. Define Intermittent Classification Functions

These functions compute the SBC indicators and classify time series based on thresholds.

In [ ]:
class Intermittent:
    @staticmethod
    def cv2(array, highest=0.05, lowest=0.05):
        '''Computes coefficient of variation squared with winsorization'''
        winsorized_array = winsorize(array, (highest, lowest))
        cv2 = (np.std(winsorized_array) / np.mean(winsorized_array)) ** 2
        return cv2
    
    @staticmethod
    def adi(array, highest=0.05, lowest=0.05):
        '''Computes average demand interval with winsorization'''
        winsorized_array = winsorize(array, (highest, lowest))
        adi = np.mean(winsorized_array)
        return adi
    
    @staticmethod
    def sddi(array, highest=0.05, lowest=0.05):
        '''Computes standard deviation of demand interval with winsorization'''
        winsorized_array = winsorize(array, (highest, lowest))
        sddi = np.std(winsorized_array)
        return sddi
    
    @staticmethod
    def compute_indicator_values(vect, threshold, perc, quant, highest, lowest):
        '''Computes indicator values for intermittent classification'''
        if isinstance(vect, np.ndarray) == False:
            try:
                vect = np.array(vect)
            except:
                raise Exception("identify_intermittent: input vect is not numeric and could not be converted")
        
        if threshold == '':
            threshold = vect[0]
            vect = vect[1:len(vect)]
        
        # Removing nan
        vect = vect[~np.isnan(vect)]
        vect = vect.astype(float)

        # Create low demand list names
        list_low_demand = ["zero", "perc_threshold"]
        for ind in ["floor_perc_quant_", "perc_quant_"]:
            list_low_demand.append(ind + str(quant).replace('0.', ''))

        for LD in list_low_demand:
            if LD == "zero":
                low_demand = 0
            elif LD == "perc_threshold":
                low_demand = perc * threshold
            elif LD == "floor_perc_quant_" + str(quant).replace('0.', ''):
                low_demand = max([0.250, 0.001 * np.quantile(vect, quant)])
            elif LD == "perc_quant_" + str(quant).replace('0.', ''):
                low_demand = perc * np.quantile(vect, quant)
            
            nzd = vect[vect > low_demand]
            k = len(nzd)
            
            if (sum(vect[vect > low_demand]) >= 2) & (k > 1):
                x = np.append([nzd[0]], [nzd[1:k] - nzd[0:(k-1)]])
                
                cv2 = Intermittent.cv2(nzd, highest, lowest)
                adi = Intermittent.adi(x, highest, lowest)
                sddi = Intermittent.sddi(x, highest, lowest)
            else:
                cv2 = np.nan
                adi = np.nan
                sddi = np.nan

            res = pd.DataFrame.from_dict({
                'type': [LD], 'k': [k], 'low_demand': [low_demand],
                'cv2': [cv2], 'adi': [adi], 'sddi': [sddi]
            })
        
        return res
    
    @staticmethod
    def classify_intermittent(df, type_val, thres_cv2_constant, thres_cv2, thres_adi, thres_sddi, min_time_cons):
        '''Classifies intermittent time series based on indicator values'''
        score_no_nan = df.dropna()

        # Regular
        mask_regular = ((score_no_nan['type'] == type_val) &
                       (score_no_nan['k'] > min_time_cons) &
                       (score_no_nan['cv2'] >= thres_cv2_constant) &
                       (score_no_nan['cv2'] < thres_cv2) &
                       (score_no_nan['adi'] < thres_adi) &
                       (score_no_nan['sddi'] < thres_sddi))
        df_regular = score_no_nan.loc[mask_regular].copy()
        df_regular['profile'] = 'regular'
        print(f'classify_intermittent: regular ids {len(df_regular)}')

        # Constant at zero
        mask_constant_zero = ((score_no_nan['type'] == type_val) &
                             (score_no_nan['k'] <= min_time_cons))
        df_constant_zero = score_no_nan.loc[mask_constant_zero].copy()
        df_constant_zero['profile'] = 'constant_zero'
        print(f'classify_intermittent: constant_zero ids {len(df_constant_zero)}')

        # Constant
        mask_constant = ((score_no_nan['type'] == type_val) &
                        (score_no_nan['k'] > min_time_cons) &
                        (score_no_nan['cv2'] < thres_cv2_constant) &
                        (score_no_nan['adi'] < thres_adi) &
                        (score_no_nan['sddi'] < thres_sddi))
        df_constant = score_no_nan.loc[mask_constant].copy()
        df_constant['profile'] = 'constant'
        print(f'classify_intermittent: constant ids {len(df_constant)}')

        # Spikes
        mask_spikes = ((score_no_nan['type'] == type_val) &
                      (score_no_nan['k'] > min_time_cons) &
                      (score_no_nan['cv2'] < thres_cv2) &
                      (score_no_nan['adi'] >= thres_adi) &
                      (score_no_nan['sddi'] < thres_sddi))
        df_spikes = score_no_nan.loc[mask_spikes].copy()
        df_spikes['profile'] = 'spikes'
        print(f'classify_intermittent: spikes ids {len(df_spikes)}')

        # Lumpy
        mask_lumpy = ((score_no_nan['type'] == type_val) &
                     (score_no_nan['k'] > min_time_cons) &
                     (score_no_nan['cv2'] >= thres_cv2) &
                     (score_no_nan['adi'] >= thres_adi) &
                     (score_no_nan['sddi'] < thres_sddi))
        df_lumpy = score_no_nan.loc[mask_lumpy].copy()
        df_lumpy['profile'] = 'lumpy'
        print(f'classify_intermittent: lumpy ids {len(df_lumpy)}')

        # Erratic
        mask_erratic = ((score_no_nan['type'] == type_val) &
                       (score_no_nan['k'] > min_time_cons) &
                       (score_no_nan['cv2'] >= thres_cv2) &
                       (score_no_nan['adi'] < thres_adi) &
                       (score_no_nan['sddi'] < thres_sddi))
        df_erratic = score_no_nan.loc[mask_erratic].copy()
        df_erratic['profile'] = 'erratic'
        print(f'classify_intermittent: erratic ids {len(df_erratic)}')

        # Unforecastable time
        mask_unforecastable_time = ((score_no_nan['type'] == type_val) &
                                   (score_no_nan['k'] > min_time_cons) &
                                   (score_no_nan['cv2'] < thres_cv2) &
                                   (score_no_nan['sddi'] >= thres_sddi))
        df_unforecastable_time = score_no_nan.loc[mask_unforecastable_time].copy()
        df_unforecastable_time['profile'] = 'unforecastable_time'
        print(f'classify_intermittent: unforecastable_time ids {len(df_unforecastable_time)}')

        # Unforecastable quantity
        mask_unforecastable_quantity = ((score_no_nan['type'] == type_val) &
                                       (score_no_nan['k'] > min_time_cons) &
                                       (score_no_nan['cv2'] >= thres_cv2) &
                                       (score_no_nan['sddi'] >= thres_sddi))
        df_unforecastable_quantity = score_no_nan.loc[mask_unforecastable_quantity].copy()
        df_unforecastable_quantity['profile'] = 'unforecastable_quantity'
        print(f'classify_intermittent: unforecastable_quantity ids {len(df_unforecastable_quantity)}')

        # Combine all profiles
        df_profiling = pd.concat([
            df_regular, df_constant_zero, df_constant, df_spikes,
            df_lumpy, df_erratic, df_unforecastable_time, df_unforecastable_quantity
        ], axis=0)
        
        return df_profiling

print("âœ… Intermittent class defined successfully")

## 3. Configuration Parameters

Set the time series and intermittent classification parameters.

In [ ]:
# Time series parameters
unique_id = 'STORE_LOCATION_ID'
date_var = 'WEEK_START_DT'
list_unique_id = [unique_id, date_var]
y = 'TOTAL_NET_SALES'
date_format = '%Y-%m-%d'

# Fabric Lakehouse configuration
# CUSTOMIZED: Lakehouse and table names for supermarket scenario
LAKEHOUSE_NAME = "ts_mmm"
INPUT_TABLE = "supermarket_net_sales_forecast_prepared"
OUTPUT_TABLE = "supermarket_net_sales_forecast_profiled"

root_path = Path.cwd().parent.parent

# Winsorizing parameters
highest = 0.05
lowest = 0.05

# Identifying intermittent time series parameters
threshold = 250
perc = 0.01
quant = 0.999

# Intermittent classification parameters (Retail demand - weekly supermarket sales)
# CUSTOMIZED: thresholds adjusted for continuous dollar-scale weekly sales data
# The SBC framework's ADI/SDDI are not meaningful for non-intermittent data (no zeros),
# so we set them above max observed values to disable intermittent/lumpy classification.
# Only the CV2 axis is active: CV2 >= 0.020 = "erratic" (higher variability stores)
thres_cv2_constant = 0.0032
thres_cv2 = 0.020  # CUSTOMIZED: P80 split â€” 12 erratic, 38 regular
thres_adi = 100000  # CUSTOMIZED: disabled (all ADI < 8428)
thres_sddi = 100000  # CUSTOMIZED: disabled (all SDDI < 75691)
# CUSTOMIZED: min_time_cons from 6 to 26 (weekly equivalent of 6 months minimum observations)
min_time_cons = 26

print("âœ… Configuration parameters set")
print(f"   - Target variable: {y}")
print(f"   - Date variable: {date_var}")
print(f"   - Unique ID: {unique_id}")
print(f"   - Input table: {INPUT_TABLE}")

print(f"   - Output table: {OUTPUT_TABLE}")
print(f"   - min_time_cons: {min_time_cons} (weekly data)")

## 4. Load Data from Lakehouse

Load the prepared data from notebook 01 (DataPreparation).

In [ ]:
# Read from Lakehouse table using Spark, then convert to Pandas
try:
    df_spark = spark.table(INPUT_TABLE)
    df = df_spark.toPandas()
except:
    print(f"âŒ Error loading data from {INPUT_TABLE}, loading from local file instead.")
    root_path = Path.cwd().parent.parent
    df = pd.read_parquet(f"{root_path}/data/{INPUT_TABLE}.parquet")

print(f"âœ… Loaded {len(df)} rows from {INPUT_TABLE}")
print(f"   Columns: {len(df.columns)}")
print(df.head(2)[[unique_id, date_var, y]])

In [ ]:
# Parse date column
df[date_var] = pd.to_datetime(df[date_var])

print(f"âœ… Date column parsed")
print(f"   Date range: {df[date_var].min()} to {df[date_var].max()}")
print(f"   Unique dates: {df[date_var].nunique()}")

## 5. Check Missing Values

In [ ]:
df_missing = df.loc[df[y].isnull(),]
print(f"Number of missing values in '{y}': {len(df_missing)}, out of {len(df)} total records.")
print(f"Percentage: {len(df_missing)/len(df)*100:.2f}%")

## 6. Define Working Dataframe

Substitute null values with 0 to represent no demand/consumption.

In [ ]:
df[y] = df[y].fillna(0)

print('Unique IDs available:', sorted(list(df[unique_id].unique()))[:5], '...')
print('Number of unique IDs:', len(list(df[unique_id].unique())))

## 7. âœ… CHECK POINT with the data scientist: visualize demand over time

Show the data scientist the plot to make sure you have correctly identified the `unique_id`.
If the plot looks wrong (e.g., unexpected grouping or missing series), revisit your `unique_id` definition and the date parsing logic.

In [ ]:
fig, ax = plt.subplots(figsize=(15, 7))
df.groupby([unique_id, date_var])[y].apply(np.mean).reset_index().pivot(
    index=date_var, columns=unique_id, values=y
).plot(ax=ax, legend=False)
ax.set_title('Net Sales per Store over Time')
ax.set_ylabel('TOTAL_NET_SALES ($)')
ax.set_xlabel('Week (Thursday start)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1e6:.1f}M'))
plt.tight_layout()
plt.show()

## 8. Compute Intermittent Indicators

Calculate CVÂ², ADI, and SDDI for each time series.

In [ ]:
# Compute indicators grouped by unique_id
df_grouped = df.groupby(unique_id)[y].apply(
    lambda y_vals: Intermittent.compute_indicator_values(
        np.array(y_vals), threshold, perc, quant, highest, lowest
    )
)

print("âœ… Intermittent indicators computed")
df_grouped.head()

In [ ]:
# Create scoring dataframe
df_scoring = df_grouped.reset_index()[[unique_id, 'type', 'k', 'low_demand', 'cv2', 'adi', 'sddi']].sort_values(by=[unique_id])

print("âœ… df_scoring created")
print(f"   Rows: {len(df_scoring)}, Columns: {len(df_scoring.columns)}")
df_scoring.head()

## 9. Indicator Statistics

In [ ]:
# Exclude NaN indicators
df_scoring_no_nan = df_scoring.dropna()
list_nan = list(set(df_scoring[unique_id]) - set(df_scoring_no_nan[unique_id]))
print('List of NaN ids:', list_nan if list_nan else 'None')

# Show indicator statistics
list_indicators = ['cv2', 'adi', 'sddi']
for ind in list_indicators:
    print(f"\n{ind.upper()} statistics:")
    print(f"  mean: {round(df_scoring[ind].mean(), 3)}")
    print(f"  max: {round(df_scoring[ind].max(), 3)}")
    print(f"  min: {round(df_scoring[ind].min(), 3)}")
    print(f"  median: {round(df_scoring[ind].median(), 3)}")

## 10. âœ… CHECK POINT with the data scientist: classify intermittent time series

In this stage, once you have computed for each `unique_id` the values of `cv2`, `adi`, `sddi` indicators, you can proceed to classify time series based on indicator thresholds.

âš ï¸ Remember that thresholds vary with respect to the type of data you are using.

In [ ]:
# Get the type value
type_val = df_scoring_no_nan['type'].unique()[0]
print(f'Type: {type_val}')

In [ ]:
# Classify time series
df_profiling = Intermittent.classify_intermittent(
    df_scoring_no_nan, 
    type_val, 
    thres_cv2_constant, 
    thres_cv2, 
    thres_adi, 
    thres_sddi, 
    min_time_cons
)

print(f"\nâœ… Classification completed")
print(f"   Total classified: {len(df_profiling)}")
df_profiling.head()

## 11. âœ… CHECK POINT with the data scientist: review profile counts and examples

If the feedback from the data scientist is negative, adjust thresholds:
1. Choose randomly 3 examples for each profile
2. Show the examples and the values of ADI and CV2
3. Ask if the profile is correct
4. If not, adjust ADI, CV2 and SDDI with a 20% change
5. Repeat until correct

In [ ]:
list_of_profiles = ['regular', 'constant_zero', 'constant', 'spikes',
                    'lumpy', 'erratic', 'unforecastable_time', 'unforecastable_quantity']

dict_profiling = {}
print("ðŸ“Š Profile Summary:")
print("-" * 50)
for c in list_of_profiles:
    dict_profiling[c] = list(
        df_profiling.loc[df_profiling.profile == c, unique_id].unique())
    count = len(dict_profiling[c])
    print(f"  {c:25s}: {count:3d} series")
    if count > 0 and count <= 5:
        print(f"    IDs: {dict_profiling[c]}")
    elif count > 5:
        print(f"    Sample IDs: {dict_profiling[c][:3]}...")
print("-" * 50)
print(f"  {'TOTAL':25s}: {len(df_profiling):3d} series")

In [ ]:
# Plot a small sample of series per profile for visual review
import random

random.seed(42)
examples_per_profile = 3

for profile_name in list_of_profiles:
    ids = dict_profiling.get(profile_name, [])
    if not ids:
        print(f"\nProfile '{profile_name}': no series")
        continue
    sample_ids = ids if len(ids) <= examples_per_profile else random.sample(ids, examples_per_profile)
    print(f"\nProfile '{profile_name}': plotting {len(sample_ids)} example series: {sample_ids}")
    
    fig, axes = plt.subplots(nrows=len(sample_ids), ncols=1, figsize=(14, 3 * len(sample_ids)), sharex=True)
    if len(sample_ids) == 1:
        axes = [axes]
    
    for ax, uid in zip(axes, sample_ids):
        ts = df.loc[df[unique_id] == uid, [date_var, y]].sort_values(date_var)
        ax.plot(ts[date_var], ts[y])
        ax.set_title(f"{profile_name} | Store {uid}")
        ax.set_ylabel(y)
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1e3:.0f}K'))
    axes[-1].set_xlabel(date_var)
    plt.tight_layout()
    plt.show()

## 12. Save Results to Lakehouse

In [ ]:
# Merge profile back to full dataset and save to Lakehouse
# df_profiling from classify_intermittent has one row per store (summary only)
# We need to merge profile labels back to the full DataFrame (5200 rows)
df_full_profiled = df.merge(
    df_profiling[[unique_id, 'profile', 'cv2', 'adi', 'sddi']], 
    on=unique_id, how='left'
)

try:
    df_profiling_spark = spark.createDataFrame(df_full_profiled)
    df_profiling_spark.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(OUTPUT_TABLE)
    print(f"✅ Saved to: {OUTPUT_TABLE}")
except Exception as e:
    print(f"❌ Error saving results to Lakehouse table: {e}")
    df_full_profiled.to_parquet(f"{root_path}/data/{OUTPUT_TABLE}.parquet")
    print(f"   Saved locally at: {root_path}/data/{OUTPUT_TABLE}.parquet")

print(f"   Profile distribution:")
print(df_full_profiled['profile'].value_counts())
print(f"\n   Rows: {len(df_full_profiled)}, Columns: {len(df_full_profiled.columns)}")


---

## âœ… Profiling Complete

**Results:**
- Regular: 38 stores (76%) â€” stable seasonal patterns
- Erratic: 12 stores (24%) â€” higher week-to-week variability (CV2 â‰¥ 0.020)
- Erratic stores: [9, 15, 16, 17, 20, 24, 25, 31, 36, 45, 47, 50]

- Erratic stores skew small/medium and are concentrated in the East region3. Erratic stores may benefit from additional smoothing features in NB05

2. Use the `supermarket_net_sales_forecast_profiled` table for downstream model selection

**Next Steps:**1. Run notebook 04 (Clustering) â€” STORE_ARCHETYPE is used directly as cluster assignment